# CVRPTW — formulacion con Pyomo + Gurobi

Modelo MILP del *Capacitated Vehicle Routing Problem with Time Windows*. Lo construimos paso a paso: SETS → PARAMETERS → VARIABLES → OBJECTIVE → CONSTRAINTS, y lo resolvemos con Gurobi.

Empezamos con un subconjunto pequeño de clientes para ver el escalado y comparar con otras soluciones o alternativas


## Formulación matemática

### Sets
- $N$: todos los nodos, con el $0$ como depósito
- $C = N \setminus \{0\}$: los clientes

### Parameters
- $c_{ij}$: distancia del nodo $i$ al $j$
- $t_{ij}$: tiempo de viaje de $i$ a $j$ (en Solomon, $t_{ij}=c_{ij}$)
- $q_i$: demanda del nodo $i$
- $Q$: capacidad del vehículo
- $a_i,\ b_i$: inicio y fin de la ventana de tiempo del nodo $i$
- $s_i$: tiempo de servicio del nodo $i$
- $K$: número máximo de vehículos
- $M$: constante grande (big-M) para las restricciones de tiempo

### Variables
- $x_{ij}\in\{0,1\}$: vale 1 si un vehículo viaja directamente de $i$ a $j$
- $w_i\ge 0$: instante en que empieza el servicio en el nodo $i$
- $u_i\ge 0$: carga acumulada del vehículo al salir del nodo $i$

### Objective
$$\min \sum_{i\in N}\sum_{j\in N,\ j\ne i} c_{ij}\, x_{ij}$$

### Constraints
1. cada cliente se visita una vez (entra una furgoneta): $\sum_{i\ne j} x_{ij} = 1 \quad \forall j\in C$
2. de cada cliente sale una furgoneta: $\sum_{j\ne i} x_{ij} = 1 \quad \forall i\in C$
3. no se usan más de $K$ vehículos: $\sum_{j\in C} x_{0j} \le K$
4. propagación del tiempo y eliminación de subtours (big-M): $w_i + s_i + t_{ij} - M\,(1-x_{ij}) \le w_j \quad \forall i\in N,\ \forall j\in C$
5. ventanas de tiempo: $a_i \le w_i \le b_i \quad \forall i\in N$
6. propagación de la carga (capacidad): $u_j \ge u_i + q_j - Q\,(1-x_{ij}) \quad \forall i\in N,\ \forall j\in C$
7. límites de carga: $q_i \le u_i \le Q \quad \forall i\in N$

## Imports

In [1]:
import sys
import os

# permite importar el paquete cvrptw desde la raiz del repo
sys.path.insert(0, os.path.abspath('..'))

import pyomo.environ as pe
import pyomo.opt as po

from cvrptw.parser import parse_solomon
from cvrptw.model import Instance

## Cargar una instancia pequeña

Leemos C101 y nos quedamos con el depósito más los primeros clientes. Reutilizamos la clase `Instance`, así que la matriz de distancias, las ventanas y las demandas ya vienen calculadas.

In [2]:
full = parse_solomon('../data/solomon/c101.txt')

# nos quedamos con el deposito (nodo 0) y los primeros n_customers clientes
n_customers = 10
small = Instance('c101_small', full.vehicle_capacity, full.nodes[:n_customers + 1])

print(small)
print('deposito:', small.depot)

Instance(name='c101_small', vehicle_capacity=200.0, size=11)
deposito: Node(index=0, x=40.0, y=50.0, demand=0.0, start_time=0.0, end_time=1236.0, service_time=0.0)


## SETS

$N$: todos los nodos (0 es el depósito)

$C$: los clientes (todos los nodos menos el depósito)

In [ ]:
model = pe.ConcreteModel('cvrptw')

nodes = list(range(small.size))     
customers = list(range(1, small.size))

model.N = pe.Set(initialize=nodes, ordered=True)
model.C = pe.Set(initialize=customers, ordered=True)

print('N:', list(model.N))
print('C:', list(model.C))

N: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
C: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
